In [1]:
"""Which method -- qPCR or sequencing -- detected each virus first each
season, and by how many days. Dumbbell/lollipop chart: all seasons plotted
on a single shared row (not one row per season -- the x-axis is already an
absolute calendar date, so a season's pair of dots naturally lands at its
own point along the timeline; a separate row per season would be redundant).
Each pair (first qPCR-positive date, first sequencing-positive date within
that season) is connected by a line. x-axis is a single shared calendar-date
axis (Oct 2022 -> Jul 2026) so
the two chart families line up for side-by-side comparison, rather than a
per-season-normalized "days since Oct 1" axis.

One chart per virus (Influenza A, Influenza B, RSV A, RSV B, SARS-CoV-2).

Season definition: Oct 1 (year Y) -> Sep 30 (year Y+1), labeled "Y-Y+1"
(used only for row grouping/labels, not for the x-axis). First qPCR date =
earliest qpcr_date among qpcr_detected=="detected" rows falling in that
season. First sequencing date = earliest metadata_date among
sequencing_detected=="detected" rows falling in that season (metadata_date
is the sequencing run date, same convention as the rest of this pipeline).

"""

#Import required packages
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = "Data_3_Correlation_Data_Final.xlsx"

COLOR_QPCR = "#2a78d6"   #  (blue)
COLOR_SEQ = "#eb6834"    #  (orange)

# Must match plot_monthly_qpcr_vs_sequencing_positivity_nodallas.py's
# timeseries charts exactly: same total figure width + same left/right
# subplot-margin fractions -> the Oct 2022-Jul 2026 axis renders at the
# identical physical/pixel width in both PNGs, so the two chart families
# line up if stacked or viewed side by side.
FIG_WIDTH_IN = 14
AXES_LEFT = 0.07
AXES_RIGHT = 0.97

# All 5 viruses' source sheets share the same raw date range (qpcr_date /
# metadata_date both run 2022-12-05/06 -> 2026-07-15)
XLIM_START = pd.Timestamp("2022-10-01")
XLIM_END = pd.Timestamp("2026-08-01")

VIRUSES = [
    {
        "sheet": "fluA",
        "title": "Influenza A",
        "out_path": "flu_a_first_detection_dumbbell.png",
    },
    {
        "sheet": "fluB",
        "title": "Influenza B",
        "out_path": "flu_b_first_detection_dumbbell.png",
    },
    {
        "sheet": "rsvA",
        "title": "RSV A",
        "out_path": "rsv_a_first_detection_dumbbell.png",
    },
    {
        "sheet": "rsvB",
        "title": "RSV B",
        "out_path": "rsv_b_first_detection_dumbbell.png",
    },
    {
        "sheet": "sars-cov-2",
        "title": "SARS-CoV-2",
        "out_path": "sarscov2_first_detection_dumbbell.png",
    },
]


def season_label(date):
    start_year = date.year if date.month >= 10 else date.year - 1
    return f"{start_year}-{start_year + 1}", pd.Timestamp(year=start_year, month=10, day=1)


def make_chart(virus):
    df = pd.read_excel(DATA_PATH, sheet_name=virus["sheet"])
    before = len(df)
    df = df[df["mismatch_flag_after_correction"].astype(str).str.lower() == "none"].copy()
    print(f"{virus['title']}: {len(df)}/{before} rows kept (mismatch_flag_after_correction == none)")

    df["qpcr_date"] = pd.to_datetime(df["qpcr_date"])
    df["metadata_date"] = pd.to_datetime(df["metadata_date"])

    # Keep the full row (not just the date) for the earliest positive each
    # season, so we can check that SAME specimen's status under the other
    # method -- distinguishes "qPCR ran on this exact sample and missed it"
    # from "qPCR simply processed the same sample a few days later."
    qpcr_hits = df[df["qpcr_detected"] == "detected"].copy()
    seq_hits = df[df["sequencing_detected"] == "detected"].copy()
    qpcr_hits[["season", "season_start"]] = qpcr_hits["qpcr_date"].apply(lambda d: pd.Series(season_label(d)))
    seq_hits[["season", "season_start"]] = seq_hits["metadata_date"].apply(lambda d: pd.Series(season_label(d)))

    qpcr_counts = qpcr_hits.groupby("season").size().rename("n_qpcr")
    seq_counts = seq_hits.groupby("season").size().rename("n_seq")

    qpcr_first_idx = qpcr_hits.groupby("season")["qpcr_date"].idxmin()
    seq_first_idx = seq_hits.groupby("season")["metadata_date"].idxmin()
    qpcr_first = qpcr_hits.loc[qpcr_first_idx].set_index("season")[
        ["sampleID", "qpcr_date", "season_start", "sequencing_detected"]
    ].rename(columns={"sampleID": "qpcr_first_sample", "sequencing_detected": "qpcr_first_sample_seq_status"})
    seq_first = seq_hits.loc[seq_first_idx].set_index("season")[
        ["sampleID", "metadata_date", "season_start", "qpcr_detected"]
    ].rename(columns={"sampleID": "seq_first_sample", "qpcr_detected": "seq_first_sample_qpcr_status"})

    seasons = qpcr_first[["season_start"]].combine_first(seq_first[["season_start"]]).join(
        qpcr_first.drop(columns="season_start"), how="outer"
    ).join(seq_first.drop(columns="season_start"), how="outer").join(qpcr_counts).join(seq_counts).sort_values("season_start")

    seasons["lead_days"] = (seasons["metadata_date"] - seasons["qpcr_date"]).dt.days  # positive = qPCR first

    def annotate(row):
        if pd.isna(row["qpcr_date"]) or pd.isna(row["metadata_date"]):
            return ""
        lead = row["lead_days"]
        if lead == 0:
            same_sample = row["qpcr_first_sample"] == row["seq_first_sample"]
            return "Same sample, same day" if same_sample else "Different samples, same day"
        if lead > 0:  # qPCR first
            missed = row["qpcr_first_sample_seq_status"] == "not detected"
            if missed:
                return f"Sequencing missed this sample\n({lead:.0f}d until a different sample was sequencing+)"
            return f"Same sample -- sequencing processed {lead:.0f}d later"
        else:  # sequencing first
            missed = row["seq_first_sample_qpcr_status"] == "not detected"
            gap = abs(lead)
            if missed:
                return f"qPCR missed this sample\n({gap:.0f}d until a different sample was qPCR+)"
            return f"Same sample -- qPCR processed {gap:.0f}d later"

    seasons["label"] = seasons.apply(annotate, axis=1)
    print(seasons[["qpcr_date", "n_qpcr", "metadata_date", "n_seq", "lead_days", "label"]])

    fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, 4.5))
    Y = 0  # single shared row -- x (actual calendar date) already separates the seasons

    for i, (season, row) in enumerate(seasons.iterrows()):
        qpcr_x, seq_x = row["qpcr_date"], row["metadata_date"]
        # Nudge apart a few days when both dots land on the exact same day,
        # so a tie shows two visible markers instead of one hiding the other.
        if pd.notna(qpcr_x) and pd.notna(seq_x) and qpcr_x == seq_x:
            qpcr_x, seq_x = qpcr_x - pd.Timedelta(days=1.5), seq_x + pd.Timedelta(days=1.5)

        if pd.notna(qpcr_x) and pd.notna(seq_x):
            line_color = COLOR_QPCR if row["lead_days"] > 0 else COLOR_SEQ
            ax.plot([qpcr_x, seq_x], [Y, Y], color=line_color, linewidth=2, alpha=0.5, zorder=2)
            mid = qpcr_x + (seq_x - qpcr_x) / 2
            ax.annotate(f"{season}\n{row['label']}", (mid, Y), xytext=(0, -35), textcoords="offset points",
                        ha="center", fontsize=9, color="#444444")
        if pd.notna(qpcr_x):
            ax.scatter([qpcr_x], [Y], color=COLOR_QPCR, s=120, zorder=3, label="First qPCR-positive" if i == 0 else None)
        if pd.notna(seq_x):
            ax.scatter([seq_x], [Y], color=COLOR_SEQ, s=120, zorder=3, label="First sequencing-positive" if i == 0 else None)

    ax.set_yticks([])
    ax.set_ylim(-0.6, 1.1)
    ax.spines["left"].set_visible(False)

    ax.set_xlim(XLIM_START, XLIM_END)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    for label in ax.get_xticklabels():
        label.set_rotation(30)
        label.set_ha("right")
    ax.set_xlabel("Date (Monthly)", fontsize=18, fontweight="bold", labelpad=15)

    ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -0.42), ncol=2, frameon=False, fontsize=16)
    
    ax.set_title(f"{virus['title']}: first qPCR-positive vs. first sequencing-positive sample, by season",
                 fontsize=18, fontweight="bold", loc="left", pad=15)
    
    fig.subplots_adjust(left=AXES_LEFT, right=AXES_RIGHT, top=0.88, bottom=0.36)

    out_dir = os.path.dirname(virus["out_path"])
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    fig.savefig(virus["out_path"], dpi=300)
    fig.savefig(virus["out_path"].replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    plt.close(fig)
    print(f"Saved {virus['out_path']}")


def main():
    for virus in VIRUSES:
        make_chart(virus)


if __name__ == "__main__":
    main()


Influenza A: 3910/3910 rows kept (mismatch_flag_after_correction == none)
           qpcr_date  n_qpcr metadata_date  n_seq  lead_days  \
season                                                         
2022-2023 2022-12-13      21    2022-12-05    111         -8   
2023-2024 2023-11-07      52    2023-10-03    422        -35   
2024-2025 2024-11-18      94    2024-10-02    517        -47   
2025-2026 2025-10-07     175    2025-10-07    352          0   

                                                       label  
season                                                        
2022-2023  qPCR missed this sample\n(8d until a different...  
2023-2024  qPCR missed this sample\n(35d until a differen...  
2024-2025  qPCR missed this sample\n(47d until a differen...  
2025-2026                        Different samples, same day  
Saved flu_a_first_detection_dumbbell.png
Influenza B: 3897/3897 rows kept (mismatch_flag_after_correction == none)
           qpcr_date  n_qpcr metadata_date  n_se